# Case study: Nikuradse's pipes, in full

In 1933 Johann Nikuradse glued sand grains of known size inside pipes and
measured the friction of the flow: 362 measurements, six roughness ratios,
Reynolds numbers from about 4,300 to 1,000,000. No exact formula is known for the
whole range. The textbook relation of Prandtl and von Kármán, with no fitted
parameter, covers the fully rough regime and explains 94 % of the variance of
these measurements.

This notebook runs the full protocol of `benchmarks/real_nikuradse.py`: a front
of formulas judged on rows kept out of the fit, then a whole roughness ratio
removed from the fit and predicted. GP_ELITE proposes short forms that fit almost
as well as the textbook relation, and long ones that fit better; none of the
short ones predicts the removed roughness as well as the textbook relation.

Every figure in the text was measured by running that script for the seeds 0 to
4 (output in `benchmarks/results_0.9/case_studies/nikuradse_seed*.txt`).

Run the cells in order with **Shift + Enter**. The two searches take about a
minute each.

*[🇫🇷 Version française](https://colab.research.google.com/github/ariel95500-create/gp-elite/blob/main/examples/case_studies/nikuradse_full.fr.ipynb)*

## 1. Install

In [ ]:
!pip install -q gp-elite

import gp_elite
print('GP_ELITE', gp_elite.__version__, 'ready.')

## 2. The data and their source

The measurements come from PMLB, the Penn Machine Learning Benchmarks
collection (dataset `nikuradse_1`), as in the quickstart notebook: the
roughness ratio `r_k` (pipe radius over grain size), the base-10 logarithm of
the Reynolds number `log_Re`, and the friction as `log10(100 * lambda)`. The
benchmarks of this project check the file by its SHA-256 fingerprint
(`benchmarks/pmlb_frozen.py`).

In [ ]:
import numpy as np
import pandas as pd

URL = ('https://github.com/EpistasisLab/pmlb/raw/master/datasets/'
       'nikuradse_1/nikuradse_1.tsv.gz')
df = pd.read_csv(URL, sep='\t', compression='gzip')

X = df[['r_k', 'log_Re']].to_numpy(float)
y = df['target'].to_numpy(float)
print(len(y), 'measurements, roughness ratios', sorted(df['r_k'].unique().tolist()))

## 3. The references

The rows are split at random, 70 % to fit and 30 % to test, always the same
way. Three references are scored on the test rows: a straight line in `r_k` and
`log_Re`; a line in `log10(r_k)` and `log_Re`, three fitted parameters; and the
Prandtl–von Kármán relation, which in this target reads
`2 - 2*log10(2*log10(r_k) + 1.74)`, with no fitted parameter at all.

In [ ]:
rng = np.random.RandomState(0)
idx = rng.permutation(len(y))
ntr = int(0.7 * len(y))
tr, te = idx[:ntr], idx[ntr:]

def r2(p, t):
    return 1 - np.mean((p - t) ** 2) / np.var(t)

def pvk(Z):     # Prandtl-von Karman, fully rough regime
    return 2.0 - 2.0 * np.log10(2.0 * np.log10(np.abs(Z[:, 0]) + 1e-12) + 1.74)

def line(basis, itr, ite):     # least squares on a basis, R2 on ite
    A = np.c_[np.ones(len(itr)), basis(X[itr])]
    coef = np.linalg.lstsq(A, y[itr], rcond=None)[0]
    return r2(np.c_[np.ones(len(ite)), basis(X[ite])] @ coef, y[ite])

logs = lambda Z: np.c_[np.log10(np.abs(Z[:, 0]) + 1e-12), Z[:, 1]]
print(f'{len(tr)} rows to fit, {len(te)} to test')
print(f'straight line, fitted          R2 test {line(lambda Z: Z, tr, te):.4f}')
print(f'line in log10(r_k), fitted     R2 test {line(logs, tr, te):.4f}')
print(f'Prandtl-von Karman, no fitting R2 test {r2(pvk(X[te]), y[te]):.4f}'
      f'   (all rows {r2(pvk(X), y):.4f})')

On the 109 test rows: the straight line R² 0.6778, the line in `log10(r_k)`
0.9346, the Prandtl–von Kármán relation 0.9456 (0.9434 on all 362 rows). That
last one is the bar.

## 4. The search, and its front

`generations=30`, four restarts, 15 % of the fitting rows held for the engine's
own validation. The table lists the Pareto front, the best formula for each
size, scored on the test rows.

In [ ]:
import time
from gp_elite import symbolic_regression

def search(Xf, yf):
    return symbolic_regression(Xf, yf, feature_names=['r_k', 'log_Re'],
                               operators='physical', normalize='auto',
                               generations=30, speed='fast',
                               validation_split=0.15, seed=0, restarts=4,
                               parallel=False)

def front(model, Zt, yt):
    rows = {}
    for e in list(model.pareto or []) + [model]:
        rows.setdefault((int(e.size), round(r2(e.predict(Zt), yt), 6)), e)
    return sorted(rows.items(), key=lambda kv: kv[0])

t0 = time.time()
m = search(X[tr], y[tr])
print(f'searched for {time.time() - t0:.0f} s\n')
print('size  R2 test  expression')
for (s, r), e in front(m, X[te], y[te]):
    print(f'{s:>4}  {r:7.4f}  {e.expression[:90]}')

**What the five seeds return.** All five start the front with the same six-node
form, `1.10108 - 0.133625 * log(r_k)`, R² 0.9357 on the test rows. Seeds 0, 2
and 4 also return a power of the roughness at seven to nine nodes:
1.559·r_k^(−1/4) − 0.0282 (seeds 0 and 4, written
`sqrt(1 / sqrt(r_k))` or `sqrt(sqrt(r_k) / r_k)`, R² 0.9447), and
0.0035 + 1.565·r_k^(−0.266) (seed 2, R² 0.9441): within 0.0015 of the
textbook relation on the test rows. Seeds 1 and 3 return forms in
`log(tanh(...))` at seven and eight nodes, R² 0.9404 to 0.9426.

The returned champion, 48 to 70 nodes, scores R² 0.980 to 0.995 on the test
rows. It follows the dependence on the Reynolds number, which the textbook
relation leaves out, in a form nobody would read as a law. On real data, the
readable answer is in the front, not in the champion. The engine may also warn
that a printed formula departs from its predictions on some rows, when a safety
net of the engine acts there; the R² in the table are those of the predictions.

## 5. The real test: a roughness never seen

`r_k` takes only six values: the data are six curves. A random split tests
points inside curves the engine has already seen. Here the roughness 126 is
removed from the fit entirely, and every form of the new front predicts it.

A law of `r_k` alone predicts a flat line along the removed curve, whose
measurements vary with the Reynolds number; R² says little there. The measure is
the level: the gap between the mean predicted and the mean measured on the
removed curve, in standard deviations of that curve.

In [ ]:
out = np.isclose(X[:, 0], 126.0)
Xin, yin, Xout, yout = X[~out], y[~out], X[out], y[out]

def bias(pred):
    return (np.mean(pred) - np.mean(yout)) / np.std(yout)

t0 = time.time()
m2 = search(Xin, yin)
print(f'searched for {time.time() - t0:.0f} s')
print(f'{len(yin)} rows to fit (five roughness ratios), {len(yout)} to predict\n')
print(f'Prandtl-von Karman: level gap {bias(pvk(Xout)):+.3f} sd')
print('size  level gap  expression')
for (s, r), e in front(m2, Xout, yout):
    print(f'{s:>4}  {bias(e.predict(Xout)):+.3f} sd   {e.expression[:70]}')

The textbook relation misses the level of the removed curve by 0.042 standard
deviations, with no choice to make. The six-node form misses it by 0.225 in all
five seeds. Over the five seeds the engine proposes 10 to 13 forms, whose gaps
range from −1.20 to +0.88. The best of each seed lies within 0.014 to 0.225 of
the level, and only two forms do better than the textbook relation, both of
seed 0 and 52 and 59 nodes long; but the best can only be picked after seeing
the removed curve. Before, nothing says which one to choose.

## 6. What the engine does not find

- **The textbook relation itself.** A logarithm whose argument holds a logarithm
  of an expression in `r_k`, the family of Prandtl and von Kármán, appears in the
  fronts of section 4 only in long forms (30 and 41 nodes for seed 1, 70 for seed
  4), and in those of section 5 at 12, 14 and 69 nodes (seed 1) and 37 and 39
  nodes (seed 3), which miss the level of the removed roughness by 0.20 to 1.20
  standard deviations, against 0.042 for the textbook relation. (The shorter run of the quickstart notebook, with other
  settings, finds one at seven nodes.) The short forms the engine proposes fit
  nearly as well on the test rows, and predict the removed roughness worse.
- **A readable law of the transitional regime.** The forms that follow the
  Reynolds number are long; this is the part of the problem that remains open.
- **What generalises.** Only a held-out test, as in section 5, says how far a
  form carries. The engine cannot say it from the fitted rows alone.

## 7. The five seeds

Sections 4 and 5 give the range over the seeds 0 to 4; the notebook runs seed 0.
The figures were measured on the development branch that follows version 0.9.0,
then again on the package of version 1.0.0, with the same results to the last
printed digit (`benchmarks/results_0.9/release_1.0/m3/`).

The full output, one file per seed: `benchmarks/results_0.9/case_studies/`. The
protocol, written before the measurements:
`benchmarks/results_0.9/PLAN_PHASE6.md`.